# WebSwipe decoder training (Colab / Kaggle)
1. Run `tools/make_colab_bundle.sh` locally and upload `colab_bundle.zip` to `MyDrive/webswipe/`.
2. Runtime > Change runtime type > T4 GPU.
3. Run all. Checkpoints are written back to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/webswipe'

In [ ]:
!rm -rf /content/ws && mkdir -p /content/ws/data
!unzip -q $ROOT/colab_bundle.zip -d /content/ws
!mv /content/ws/*_features.npz /content/ws/data/
%cd /content/ws
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Stage 1: reproduce Shen et al. on their mid-air data (random 95/5 split, as in the paper code)

In [ ]:
!python -m webswipe.decoder.train --data data/shen_features.npz --epochs 200 --eval-every 5 --batch 64 --lr 1e-4 --out $ROOT/checkpoints/shen_d1_random.pt

## Leave-one-subject-out (harder, more honest split)
Change `--test-subject 0:<id>` for each of the 9 subjects.

In [ ]:
!python -m webswipe.decoder.train --data data/shen_features.npz --split subject --test-subject 0:0 --epochs 100 --eval-every 5 --out $ROOT/checkpoints/shen_d1_loso0.pt

## Stage 2: fine-tune on our webcam data (after D3 is recorded)

In [ ]:
# !python -m webswipe.decoder.train --data data/webcam_features.npz --split subject --test-subject 0:<id> \
#     --init $ROOT/checkpoints/shen_d1_random.pt --lr 3e-5 --epochs 50 --out $ROOT/checkpoints/ft_webcam.pt